# Used Vehicle Price Prediction: A Hedonic Pricing Model for Craigslist Listings

**Final Project Proposal** 

**Alec Blevins** 

**STA 334 - Statistical Consulting**

**October 16, 2026**

## 1. Introduction

### 1.1 Business context

A used car dealership must decide what asking price to post for each vehicle in its inventory. Post too high and cars sit on the lot, accruing holding costs and losing value to depreciation. Post too low and margin is lost on a transaction that would have cleared at a higher price. A dealership therefore needs an empirical benchmark: *given a vehicle's observable characteristics, what asking price is typical for comparable listings?*

This project builds such a benchmark. Using a large sample of Craigslist used-vehicle listings, we estimate a **hedonic pricing model** - a regression that predicts a vehicle's asking price from its attributes - and quantify the implicit dollar value of each attribute. The hedonic approach is the standard method in real estate and used-vehicle modeling for decomposing the price of a good into the prices of its components. 

### 1.2 Client question
> *"Given a vehicle's characteristics, what asking price is typical for comparable listings, and which attributes are associated with higher posted prices?"*

The client is a hypothetical used-car dealership pricing its inventory. The deliverable is a model that answers this question, a ranked list of the attributes that matter most, and a flag for listings whose asking price deviates substantially from comparable inventory. 

### 1.3 Statistical goals

1. **Estimate a hedonic pricing model** on Craigslist used-vehicle listings. 
2. **Quantify how much each attribute contributes** to posted price, controlling for the others. 
3. **Test whether dealer listings price differently** from private-party listings for otherwise-comparable vehicles. 
4. **Flag listings whose asking price deviates** substantially from what comparable inventory posts. 

### 1.4 A note on the response variable

The `price` field in the dataset records the seller's *asking price*, not the price at which the vehicle sold. Craigslist listings are not transactions, and listings that never sold are indistinguishable from listings that sold at or near the asking price. Coefficients therefore desscribe how *posted* prices vary with vehicle characteristics, rather than the market-clearing transaction price. 

### 1.5 Preliminary findings

Exploratory analysis (documented in `01_eda.ipynb`) identified two structural challenges in the raw data: 

1. **Placeholder values.** Roughly 11% of listings report a price of $1,000 or less, and the maximum reported price is 3.7 billion. These are data entry artifacts, not real listings. 
2. **Two mixed populations.** The dataset contains both *normal listings*, whose prices reflect the vehicle's age, mileage, and type, and *distressed listings* - salvage, rebuilds, project cars, "mechanic specials" - whose prices reflect what is left of a broken vehicle. A single model fit to the combined data fails because the price-generating pricess is different in the two populations. 

These findings motivate the cleaning and feature-engineering pipeline documented in `02_features.ipynb` and summarized in Section 2. 

## 2. Methodology

### 2.1 Data

**Source.** The Craigslist Used Cars dataset, publicly available on Kaggle (`https://github.com/AustinReese/UsedVehicleSearch.git`). The raw file contains **426,880 listings** and **26 columns**, including price, year, manufacturer, model, condition, cylinders, fuel, odometer, title status, transmission, drive type, size, vehicle type, paint color, state, latitude, longitude, and posting date.

**Data availability.** The dataset is downloaded from Kaggle and stored locally. All code is written in Python 3.14 using `pandas`, `numpy`, `matplotlib`, and `seaborn` for analysis. `scikit-learn` and `statsmodels` will be considered for modeling. 

### 2.2 Response variable

The modeling response is `log_price = log(price)`. Price is right-skewed; the log transform stabilizes variance and makes OLS coefficients interpretable as approximate percentage effects. The log-transformed target is approximately normal after the cleaning step.

### 2.3 Predictors

The raw dataset provides two categories of predictors: **numeric** (year, odometer) and **categorical** (manufacturer, model, condition, cylinders, fuel, title status, transmission, drive, size, type, paint color, state).

Three derived predictors are constructed: 
| Feature | Type | Derivation |
|---|---|---|
| `is_dealer` | Binary | Craigslist URL path (`/ctd/` = dealer, `/cto/` = private) |
| `census_region` | 4-level categorical | US Census region mapped from `state` |
| `dist_to_major_metro` | Numeric (miles) | Haversine distance to nearest of 20 US metro centroids |

`year`, `odometer`, and `is_dealer` are the primary continuous and structural predictors. `manufacturer`, `type`, `fuel`, `drive`, `condition`, and `census_region` are the primary categorical predictors. `model` (15,141 unique values) and `region` (404 unique values) are excluded due to cardinality. 

### 2.4 Data cleaning

The raw dataset is filtered through five sequential steps, each targeting a specific distress signal identified in the EDA: 
| # | Filter | Rationale |
|---|---|---|
| 1 | `title_status == "clean"` | Removes salvage, rebuilt, lien, missing, parts-only titles |
| 2 | `1,000 ≤ price ≤ 150,000` | Removes placeholder prices and extreme outliers |
| 3 | `2008 ≤ year ≤ 2022` | Restricts to modern vehicles with stable depreciation |
| 4 | `1,000 ≤ odometer ≤ 180,000` | Removes zero-mile placeholders and implausible mileages |
| 5 | `condition ∉ {fair, salvage}`; nulls retained | Removes explicitly-declared bad condition |

Filtering reduces the same to **257,781 listings (60.4% of raw)**. The retention rate is higher than the sum of individual signal counts would suggest, because distress signals overlap heavily. 

### 2.5 Handling missing values

Missing values in categorical predictors (`condition`, `cylinders`, `drive`, `paint_color`, `size`, `type`) are retained as an explicit `"Unknown"` level rather than dropped. Retaining nulls as a level allows the model to estimate whether missingness itself carries signal. 

### 2.6 Models

We consider a sequence of models of increasing flexibility: 

1. **OLS hedonic regression.** `log_price` regressed on all numeric and one-hot-encoded categorical predictors. This is an interpretable model and may serve as a baseline: coefficients read directly as approximate percentage effects on asking price. 
2. **Regularized regression.** Ridge and Lasso, fit on the same design matrix. Lasso performs variable selection, which may be useful with many one-hot columns. 
3. **Gradient boosting.** `HistGradientBoostingRegressor` or XGBoost, which handle nonlinearity. Compared to OLS on RMSE and R-Squared; can be interpreted via SHAP values. 
4. **Dealer-only sensitivity model.** Can be refit on the 89.7% dealer subsample without `is_dealer`. We can then compare to the full-sample model. 

### 2.7 Validation

**Split.** A time-ordered split: the first 70% of listings by posting date can be used for training, the next 15% for validation and hyperparameter tuning, and the final 15% for a single held-out test. Random splits shouldn't be used because the data is temporal and random splits leak future information. 

**Evaluation metrics.**
- **RMSE** on `log_price`, comparable across models.
- **R²** on both train and test sets, to detect overfitting.
- **Residual plots** to check for structure the model misses.
- **SHAP values** for the tree-based model, to interpret feature importance and direction of effects.

### 2.8 Known limitations
- **Asking prices, not transaction prices.** The model predicts posted prices, not negotiated sale prices. 
- **Unobserved mechanical condition.** The model cannot observe whether a vehicle runs, whether the transmission works, or whether the frame is damaged. Listings whose prices reflect undisclosed mechanical problems will appear as residual anomalies.
- **Dealer-dominated sample.** After filtering, 89.7% of listings are dealer postings. Thus, the model is best interpreted as a dealer-inventory pricing model with a private-party comparison subsample.

### 2.9 Completed preliminary work

Two notebooks have been completed as of this final project proposal: 

**`01_eda.ipynb` — Exploratory Data Analysis**
- Profiled the raw dataset.
- Characterized the target distribution, identifying the appropriate log transformation.
- Crosstabulated `title_status` and `condition` against price buckets, establishing that low-price listings are disproportionately distressed. 
- Quantified six distress signals and their overlap. 

**`02_features.ipynb` — Feature Engineering**
- Applied five sequential filters. 
- Constructed `is_dealer`, `census_region`, and `dist_to_major_metro`.
- Investigated and corrected implausible geographic coordinates. 
- Dropped identifier, redundant, and constant columns.
- Saved the modeling subset

## 3. Project Plan

### 3.1 Deliverables
1. **`03_modeling.ipynb`** — model fitting, comparison, and evaluation.
2. **`04_analysis.ipynb`** — residual analysis, over/underpriced listing detection, and business framing.
3. **Final written report**
4. **Final presentation**

### 3.2 Timeline
| Weeks | Dates | Tasks |
|---|---|---|
| 7-8 | Oct 12–21 | Submit proposal due Oct 16th. Set up `03_modeling.ipynb` scaffold. Split the modeling subset by posting date. |
| 9 | Oct 26-28 | Fit baseline, OLS, and Ridge/Lasso models. Prepare progress report due Nov 5th. |
| 10 | Nov 2–4 | Fit gradient boosting. Compare to OLS on held-out RMSE and R². Compute SHAP values. Submit progress report. |
| 11 | Nov 9–11 | Dealer-only sensitivity model and residual analysis. Prepare final analysis and report. |
| 12-13 | Nov 16–25 | Draft project report, finalize project presentation. |
| 14-15 | Nov 30–Dec 9 | Finalize deliverables, github, readme, submit. |

## 4. References

### Data sources
- **Craigslist Used Cars dataset** (Kaggle). Publicly available at `kaggle.com/datasets/austinreese/craigslist-carstrucks-data`. Accessed September 2026.
- **US Census Bureau region classification.** Standard four-region classification of US states.

### Use of AI Tools
During the preparation of this work, DeepSeek-V4.1-Flash was used to assist with brainstorming the project scope, refining statistical methodology, and structuring this proposal document. All output was reviewed and edited by the author. 

